# Лабораторная работа №1
## Тема: Предобработка данных. Знакомство с pandas
**Датасет:** [Stroke Prediction Dataset на Kaggle](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)  
**Цель:** Изучить базовые методы предобработки данных: обработку пропущенных значений, нормализацию числовых признаков и кодирование категориальных данных.

In [1]:
import pandas as pd
import numpy as np

# Настройка для красивого отображения всех колонок таблицы
pd.set_option('display.max_columns', None)

In [2]:
# Загрузка датасета в DataFrame
# Если вы перенесли файл в папку ml_lab1:
df = pd.read_csv('stroke_data.csv')

# (Если файл остался в папке archive, используйте: df = pd.read_csv('archive/stroke_data.csv'))

# Вывод первых 5 строк (Пункт 2 практического задания)
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
# Получение информации о количестве строк, столбцах и типах данных
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


## 1. Анализ и обработка пропущенных значений
Подсчитаем количество пропущенных значений в каждом признаке датасета с помощью методов `isnull().sum()`.

In [4]:
# Подсчет количества NaN в каждом столбце
missing_values = df.isnull().sum()
print("Количество пропусков по колонкам:")
print(missing_values)

Количество пропусков по колонкам:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


In [5]:
# Посмотрим на уникальные значения в колонке smoking_status
df['smoking_status'].value_counts()

smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64

In [6]:
# Заменяем скрытые пропуски 'Unknown' на официальный признак отсутствия данных NaN
df['smoking_status'] = df['smoking_status'].replace('Unknown', np.nan)

# Проверяем обновленное количество пропусков
df.isnull().sum()

id                      0
gender                  0
age                     0
hypertension            0
heart_disease           0
ever_married            0
work_type               0
Residence_type          0
avg_glucose_level       0
bmi                   201
smoking_status       1544
stroke                  0
dtype: int64

In [7]:
# 1. Для категориального признака вычисляем моду (самое частое значение)
smoking_mode = df['smoking_status'].mode()[0]
print(f"Мода для smoking_status: {smoking_mode}")

# 2. Для числового признака вычисляем медиану (устойчива к аномалиям и выбросам)
bmi_median = df['bmi'].median()
print(f"Медиана для bmi: {bmi_median:.2f}")

# Заполняем пропуски значениями
df['smoking_status'] = df['smoking_status'].fillna(smoking_mode)
df['bmi'] = df['bmi'].fillna(bmi_median)

Мода для smoking_status: never smoked
Медиана для bmi: 28.10


In [8]:
# Проверка 1: суммарное количество пропусков во всем датафрейме должно быть 0
print("Осталось пропусков после заполнения:")
print(df.isnull().sum())

# Проверка 2: посмотрим на строку с индексом 1 (где раньше в bmi был NaN)
print("\nПроверка строки 1 (ранее содержавшей NaN в bmi):")
df.loc[[1], ['gender', 'age', 'bmi', 'smoking_status']]

Осталось пропусков после заполнения:
id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

Проверка строки 1 (ранее содержавшей NaN в bmi):


,gender,age,bmi,smoking_status
1,Female,61.0,28.1,never smoked


In [9]:
# Удаление столбца 'id' из датафрейма
df.drop('id', axis='columns', inplace=True)
print("Столбец 'id' успешно удален. Текущие колонки:")
print(df.columns.tolist())

Столбец 'id' успешно удален. Текущие колонки:
['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']


## 2. Нормализация данных
Приведем непрерывные числовые признаки (`age`, `avg_glucose_level`, `bmi`) к единому масштабу с помощью `StandardScaler` (Z-оценка), чтобы среднее значение равнялось 0, а стандартное отклонение — 1.

In [10]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Выбираем непрерывные числовые признаки для масштабирования
num_cols = ['age', 'avg_glucose_level', 'bmi']

# Создаем объект масштабирования (Z-score стандартизация)
scaler = StandardScaler()

# Вычисляем параметры формулы (fit) и применяем преобразование (transform)
df[num_cols] = scaler.fit_transform(df[num_cols])

# Выводим первые 5 строк нормализованных признаков
df[num_cols].head()

,age,avg_glucose_level,bmi
0,1.051434,2.706375,1.005086
1,0.786070,2.121559,-0.098981
2,1.626390,-0.005028,0.472536
3,0.255342,1.437358,0.719327
4,1.582163,1.501184,-0.631531


In [11]:
# Статистическое описание нормализованных признаков
df[num_cols].describe().round(2)

,age,avg_glucose_level,bmi
count,5110.00,5110.00,5110.00
mean,0.00,0.00,-0.00
std,1.00,1.00,1.00
min,-1.91,-1.13,-2.41
25%,-0.81,-0.64,-0.66
50%,0.08,-0.31,-0.10
75%,0.79,0.18,0.51
max,1.71,3.66,8.93


## 3. Преобразование категориальных признаков в численные (One-Hot Encoding)
Преобразуем строковые признаки в бинарные столбцы методом `pd.get_dummies()`.  
Используем флаг `drop_first=True`, чтобы избежать мультиколлинеарности («ловушки фиктивных переменных») и предотвратить переобучение будущей модели.

In [12]:
# Автоматически находим все строковые (категориальные) признаки
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Категориальные признаки для кодирования:", cat_cols)

Категориальные признаки для кодирования: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


C:\Users\andre\AppData\Local\Temp\ipykernel_21612\3453055782.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [13]:
# Применяем One-Hot Encoding с отбрасыванием первой фиктивной переменной
# Параметр dtype=int преобразует True/False в аккуратные числа 1 и 0
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# Проверяем получившийся датасет
print(f"Новая размерность датасета: {df.shape} (строк, столбцов)")
df.head()

Новая размерность датасета: (5110, 16) (строк, столбцов)


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_never smoked,smoking_status_smokes
0,1.051434,0,1,2.706375,1.005086,1,1,0,1,0,1,0,0,1,0,0
1,0.786070,0,0,2.121559,-0.098981,1,0,0,1,0,0,1,0,0,1,0
2,1.626390,0,1,-0.005028,0.472536,1,1,0,1,0,1,0,0,0,1,0
3,0.255342,0,0,1.437358,0.719327,1,0,0,1,0,1,0,0,1,0,1
4,1.582163,1,0,1.501184,-0.631531,1,0,0,1,0,0,1,0,0,1,0


In [14]:
# Убеждаемся, что в датасете больше нет строк (только int и float) и нет пропусков
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          5110 non-null   float64
 1   hypertension                 5110 non-null   int64  
 2   heart_disease                5110 non-null   int64  
 3   avg_glucose_level            5110 non-null   float64
 4   bmi                          5110 non-null   float64
 5   stroke                       5110 non-null   int64  
 6   gender_Male                  5110 non-null   int64  
 7   gender_Other                 5110 non-null   int64  
 8   ever_married_Yes             5110 non-null   int64  
 9   work_type_Never_worked       5110 non-null   int64  
 10  work_type_Private            5110 non-null   int64  
 11  work_type_Self-employed      5110 non-null   int64  
 12  work_type_children           5110 non-null   int64  
 13  Residence_type_Urban         

In [15]:
from sklearn.model_selection import train_test_split

# Разбиваем датасет: 80% на обучение, 20% на тест (random_state для воспроизводимости)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {train_df.shape}")
print(f"Размер тестовой выборки: {test_df.shape}")

Размер обучающей выборки: (4088, 16)
Размер тестовой выборки: (1022, 16)


In [16]:
# Сохранение готового датасета в новый CSV файл
df.to_csv("processed_stroke_data.csv", index=False)
print("Файл processed_stroke_data.csv успешно сохранен!")

Файл processed_stroke_data.csv успешно сохранен!
